# Prediksi `tc_percentage` kabupaten/kota di Sumatera

Analisis ini mengisi nilai `tc_resmi_pct` yang kosong pada 2025 dengan estimasi per kabupaten/kota. Target berskala persen (misalnya 55,1 berarti 55,1%). Angka resmi di atas 100% dipertahankan sesuai kamus data.

Model memakai fitur wilayah tahun 2025, graf batas administratif, dan TC dari sebelum 2025. Jalankan notebook dari folder proyek dengan kernel Python yang memiliki `pandas`, `numpy`, `openpyxl`, `scikit-learn`, dan `torch` (CPU cukup).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.base import BaseEstimator, clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn

torch.set_num_threads(1)

DATA_DIR = Path("data")
YEARS = (2023, 2024, 2025)
TARGET = "tc_resmi_pct"
HISTORY = "tc_tahun_sebelumnya"
NODE_COLUMNS = [
    "kepadatan_jiwa_km2",
    "penduduk_miskin_pct",
    "rt_hunian_layak_pct",
    "puskesmas_per_100rb_penduduk",
    "wilayah_kota",
]
FEATURES = ["log_penduduk", "log_kepadatan", *NODE_COLUMNS[1:]]


def load_year(year):
    path = DATA_DIR / f"TRACK-TB_Join_Sumatera_{year}.xlsx"
    frame = pd.read_excel(path, sheet_name=f"JOIN_{year}", dtype={"kode_bps": str})
    frame = frame.set_index("kode_bps").sort_index()
    if not frame.index.is_unique or frame.index.hasnans or not frame["tahun"].eq(year).all():
        raise ValueError(f"Kunci atau tahun pada {path.name} tidak valid.")

    nodes = pd.read_excel(
        path, sheet_name="GNN_NODES", dtype={"kode_bps": str},
        usecols=["kode_bps", "node_id", "tahun", "derajat_node", *NODE_COLUMNS],
    ).set_index("kode_bps").sort_index()
    if not nodes.index.equals(frame.index) or not nodes["tahun"].eq(year).all():
        raise ValueError(f"Kunci atau tahun GNN_NODES pada {path.name} tidak cocok.")
    tabular = frame.assign(wilayah_kota=frame["jenis_wilayah"].eq("Kota").astype(int))
    np.testing.assert_allclose(nodes[NODE_COLUMNS], tabular[NODE_COLUMNS], rtol=1e-9)

    edges = pd.read_excel(
        path, sheet_name="GNN_EDGES", dtype={"kode_bps_a": str, "kode_bps_b": str},
        usecols=["src_node_id", "dst_node_id", "kode_bps_a", "kode_bps_b"],
    )
    codes_by_node_id = pd.Series(nodes.index, index=nodes["node_id"])
    if not codes_by_node_id.index.is_unique:
        raise ValueError(f"node_id ganda pada {path.name}.")
    for id_col, code_col in (("src_node_id", "kode_bps_a"), ("dst_node_id", "kode_bps_b")):
        if not edges[id_col].map(codes_by_node_id).eq(edges[code_col]).all():
            raise ValueError(f"Kode BPS edge tidak cocok pada {path.name}.")

    a = frame.index.get_indexer(edges["kode_bps_a"])
    b = frame.index.get_indexer(edges["kode_bps_b"])
    pairs = pd.MultiIndex.from_arrays([np.minimum(a, b), np.maximum(a, b)])
    if (a < 0).any() or (b < 0).any() or (a == b).any() or not pairs.is_unique:
        raise ValueError(f"Ada edge tidak valid pada {path.name}.")
    adjacency = np.zeros((len(frame), len(frame)), dtype=float)
    adjacency[a, b] = adjacency[b, a] = 1
    np.testing.assert_array_equal(adjacency.sum(axis=1), nodes["derajat_node"])

    frame[NODE_COLUMNS] = nodes[NODE_COLUMNS]
    frame["log_penduduk"] = np.log1p(frame["penduduk_jiwa"])
    frame["log_kepadatan"] = np.log1p(frame["kepadatan_jiwa_km2"])
    if not np.isfinite(frame[FEATURES]).to_numpy().all():
        raise ValueError(f"Ada fitur kosong atau tidak valid pada {year}.")
    return frame, adjacency


(data23, adjacency), (data24, adjacency24), (data25, adjacency25) = (
    load_year(year) for year in YEARS
)
for frame, year_graph in ((data24, adjacency24), (data25, adjacency25)):
    if not frame.index.equals(data23.index):
        raise ValueError("Daftar kode BPS berubah antartahun.")
    np.testing.assert_array_equal(year_graph, adjacency)

degree = adjacency.sum(axis=1)
neighbors = adjacency + np.diag(degree == 0)
neighbor_average = neighbors / neighbors.sum(axis=1, keepdims=True)
with_self_loops = adjacency + np.eye(len(adjacency))
inv_sqrt_degree = with_self_loops.sum(axis=1) ** -0.5
gcn_adjacency = inv_sqrt_degree[:, None] * with_self_loops * inv_sqrt_degree[None, :]

frames = {2023: data23, 2024: data24, 2025: data25}
coverage = pd.concat(
    {year: frame.groupby("provinsi")[TARGET].count() for year, frame in frames.items()},
    axis=1,
)
print(f"Graf: {len(adjacency)} simpul, {int(adjacency.sum() / 2)} edge, "
      f"{int((degree == 0).sum())} simpul tanpa tetangga.")
print("\nJumlah TC resmi per provinsi dan tahun")
print(coverage.to_string())
print("\nTotal TC resmi:", coverage.sum().to_dict())

Graf: 154 simpul, 315 edge, 8 simpul tanpa tetangga.

Jumlah TC resmi per provinsi dan tahun
                           2023  2024  2025
provinsi                                   
Aceh                         23     0     0
Bengkulu                     10     0     0
Jambi                        11    11    11
Kepulauan Bangka Belitung     7     7     0
Kepulauan Riau                7     7     0
Lampung                      15    15    15
Riau                         12    12    12
Sumatera Barat               19    19    19
Sumatera Selatan             17     0     0
Sumatera Utara               33     0     0

Total TC resmi: {2023: 154, 2024: 71, 2025: 57}


### Fitur dan graf

Lima variabel dasar berasal dari `GNN_NODES`. Jumlah penduduk dari `JOIN_*` dan TC tahun sebelumnya melengkapi fitur. Nilai fitur dasar pada kedua lembar diperiksa kesesuaiannya. `GNN_EDGES` menghubungkan kabupaten/kota yang berbagi batas administratif. Model memakai edge dua arah dengan bobot yang sama.

- **Ridge tabular** menjadi pembanding dengan fitur wilayah sendiri.
- **Ridge + fitur tetangga** menambahkan rata-rata setiap fitur di wilayah yang berbatasan dan jumlah tetangga. Wilayah tanpa tetangga memakai fitur sendiri sebagai pengganti rata-rata.
- **GCN** (Graph Convolutional Network) menyebarkan fitur melalui graf dalam dua lapis, dengan jalur tambahan untuk fitur wilayah sendiri. Model ini memakai normalisasi graf dan *self-loop* berdasarkan [Kipf dan Welling](https://arxiv.org/abs/1609.02907), yang diadaptasi untuk regresi.

Label TC tahun berjalan, kolom notifikasi dan estimasi insiden, serta status label dikeluarkan dari input. Graf dan fitur semua wilayah boleh dilihat karena wilayah yang akan diprediksi sudah diketahui. Hanya label dari provinsi yang masuk lipatan latih yang dipakai untuk menyesuaikan model.

In [2]:
class GraphNetwork(nn.Module):
    def __init__(self, feature_count, hidden_units):
        super().__init__()
        self.hidden = nn.Linear(feature_count, hidden_units)
        self.graph_output = nn.Linear(hidden_units, 1)
        self.local_output = nn.Linear(feature_count, 1)

    def forward(self, features, graph):
        hidden = torch.relu(self.hidden(graph @ features))
        return (self.graph_output(graph @ hidden) + self.local_output(features)).squeeze(-1)


class GCNRegressor(BaseEstimator):
    def __init__(self, normalized_adjacency, hidden_units=16, epochs=150, seed=42):
        self.normalized_adjacency = normalized_adjacency
        self.hidden_units = hidden_units
        self.epochs = epochs
        self.seed = seed

    def fit(self, x, y):
        self.scaler_ = StandardScaler().fit(x.loc[y.index])
        self.graph_ = torch.as_tensor(self.normalized_adjacency, dtype=torch.float32)
        features = torch.as_tensor(self.scaler_.transform(x), dtype=torch.float32)
        train_positions = torch.as_tensor(x.index.get_indexer(y.index))
        self.target_mean_ = float(y.mean())
        self.target_scale_ = float(y.std(ddof=0)) or 1.0
        target = torch.as_tensor(
            (y.to_numpy() - self.target_mean_) / self.target_scale_,
            dtype=torch.float32,
        )
        torch.manual_seed(self.seed)
        self.network_ = GraphNetwork(x.shape[1], self.hidden_units)
        optimizer = torch.optim.Adam(self.network_.parameters(), lr=0.01, weight_decay=0.03)
        loss_function = nn.MSELoss()
        self.network_.train()
        for _ in range(self.epochs):
            optimizer.zero_grad()
            predicted = self.network_(features, self.graph_)
            loss = loss_function(predicted[train_positions], target)
            loss.backward()
            optimizer.step()
        return self

    def predict(self, x):
        features = torch.as_tensor(self.scaler_.transform(x), dtype=torch.float32)
        self.network_.eval()
        with torch.inference_mode():
            prediction = self.network_(features, self.graph_).numpy()
        return self.target_mean_ + self.target_scale_ * prediction.astype(float)


def design(frame, previous_tc):
    x = frame[FEATURES].assign(**{HISTORY: previous_tc})
    if not x.index.equals(data23.index) or not np.isfinite(x).to_numpy().all():
        raise ValueError("Input graf harus memuat fitur dan riwayat semua wilayah sesuai urutan node.")
    return x


def neighbor_features(x):
    averaged = pd.DataFrame(
        neighbor_average @ x.to_numpy(), index=x.index,
        columns=[f"tetangga_{column}" for column in x.columns],
    )
    return x.join(averaged).assign(log_jumlah_tetangga=np.log1p(degree))


def fit_model(name, x, y):
    estimator = candidates[name]
    if estimator is None:
        return None
    if name == "GCN":
        return clone(estimator).fit(x, y)
    if name == "Ridge + fitur tetangga":
        x = neighbor_features(x)
    return clone(estimator).fit(x.loc[y.index], y)


def predict_model(name, model, x):
    if model is None:
        return x[HISTORY]
    if name == "Ridge + fitur tetangga":
        x = neighbor_features(x)
    return pd.Series(np.maximum(model.predict(x), 0), index=x.index)


def metrics(actual, predicted):
    return {
        "n": len(actual),
        "MAE_pp": mean_absolute_error(actual, predicted),
        "RMSE_pp": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
    }


candidates = {
    "Median data latih": DummyRegressor(strategy="median"),
    "TC resmi tahun terakhir": None,
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=20)),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, max_depth=4, min_samples_leaf=6, random_state=42
    ),
    "Ridge + fitur tetangga": make_pipeline(StandardScaler(), Ridge(alpha=20)),
    "GCN": GCNRegressor(gcn_adjacency),
}

### Validasi antarprovinsi 2024

Pemilihan model menggunakan MAE dari validasi *leave-one-province-out* pada 71 wilayah berlabel 2024. TC resmi 2023 tersedia sebagai fitur riwayat untuk semua wilayah. Arsitektur dan parameter model ditetapkan sebelum penilaian 2025.

In [3]:
y24 = data24[TARGET].dropna()
x24_all = design(data24, data23[TARGET])
province24 = data24.loc[y24.index, "provinsi"]
folds = list(LeaveOneGroupOut().split(y24, groups=province24))

development_scores = {}
for name in candidates:
    out_of_fold = pd.Series(index=y24.index, dtype=float)
    for train, valid in folds:
        model = fit_model(name, x24_all, y24.iloc[train])
        validation_areas = y24.index[valid]
        out_of_fold.loc[validation_areas] = predict_model(name, model, x24_all).loc[validation_areas]
    development_scores[name] = metrics(y24, out_of_fold)

development = pd.DataFrame.from_dict(development_scores, orient="index")
development = development.sort_values("MAE_pp", kind="stable")
selected = development.index[0]
print(development.round(3).to_string())
print(f"\nModel terpilih berdasarkan MAE validasi 2024: {selected}")

                          n  MAE_pp  RMSE_pp     R2
Ridge                    71  11.541   15.438  0.596
GCN                      71  11.656   16.005  0.565
Ridge + fitur tetangga   71  12.013   15.514  0.592
Random Forest            71  12.254   15.620  0.586
TC resmi tahun terakhir  71  14.148   17.618  0.473
Median data latih        71  18.170   25.021 -0.062

Model terpilih berdasarkan MAE validasi 2024: Ridge


### Uji 2025

Uji memakai label dari 57 kabupaten/kota pada empat provinsi. Kedua skenario memakai wilayah uji yang sama.

1. **TC 2024 tersedia.** Model dilatih pada 71 target 2024 dan memakai TC resmi 2024 sebagai riwayat wilayah uji. Jika tetangga memiliki riwayat kosong, model mengisi riwayat itu dari estimasi 2024.
2. **TC 2024 disimulasikan kosong.** Untuk setiap provinsi uji, semua label 2024 dari provinsi itu dihapus dari pelatihan dan dari input riwayat. Model memperkirakan TC 2024 yang kosong dari riwayat 2023. Baru setelah itu model memprediksi TC 2025, termasuk melalui hubungan tetangga lintas provinsi.

Label 2025 hanya dipakai menghitung galat. MAE dan RMSE dilaporkan dalam poin persentase. Keenam provinsi dengan label 2025 kosong berada di luar cakupan pengukuran galat.

In [4]:
y25 = data25[TARGET].dropna()
province25 = data25.loc[y25.index, "provinsi"]
official_2024 = data24[TARGET]
if official_2024.loc[y25.index].isna().any():
    raise ValueError("Skenario uji satu tahun membutuhkan TC resmi 2024.")


def forecast_2025(name, model, history_2024):
    estimated_2024 = predict_model(name, model, x24_all)
    x25 = design(data25, history_2024.fillna(estimated_2024))
    return predict_model(name, model, x25)


predictions_2025 = {}
holdout_scores = {}
holdout_predictions = {}
for name in candidates:
    model = fit_model(name, x24_all, y24)
    predictions_2025[name] = forecast_2025(name, model, official_2024)
    one_year = predictions_2025[name].loc[y25.index]

    two_years = pd.Series(index=y25.index, dtype=float)
    for province in province25.unique():
        train = province24.ne(province)
        area = province25.index[province25.eq(province)]
        model = fit_model(name, x24_all, y24.loc[train])
        masked_history = official_2024.mask(data24["provinsi"].eq(province))
        two_years.loc[area] = forecast_2025(name, model, masked_history).loc[area]

    for scenario, prediction in (("TC 2024 tersedia", one_year), ("TC 2024 kosong", two_years)):
        holdout_scores[scenario, name] = metrics(y25, prediction)
        holdout_predictions[scenario, name] = prediction

holdout = pd.DataFrame.from_dict(holdout_scores, orient="index")
holdout.index = pd.MultiIndex.from_tuples(holdout.index, names=["skenario", "model"])
print(holdout.sort_index(level="skenario", sort_remaining=False).round(3).to_string())

province_errors = pd.DataFrame({
    scenario: (y25 - holdout_predictions[scenario, selected]).abs().groupby(province25).mean()
    for scenario in ("TC 2024 tersedia", "TC 2024 kosong")
})
print(f"\nMAE {selected} per provinsi (poin persentase)")
print(province_errors.round(2).to_string())

                                           n  MAE_pp  RMSE_pp     R2
skenario         model                                              
TC 2024 kosong   Median data latih        57  15.603   24.402 -0.194
                 TC resmi tahun terakhir  57  16.364   20.710  0.140
                 Ridge                    57  12.476   18.360  0.324
                 Random Forest            57  12.376   16.826  0.432
                 Ridge + fitur tetangga   57  11.417   16.797  0.434
                 GCN                      57  15.422   21.900  0.038
TC 2024 tersedia Median data latih        57  15.468   24.055 -0.160
                 TC resmi tahun terakhir  57   8.553   11.488  0.735
                 Ridge                    57  10.214   12.750  0.674
                 Random Forest            57  13.476   17.843  0.361
                 Ridge + fitur tetangga   57   9.568   12.350  0.694
                 GCN                      57  12.262   14.750  0.564

MAE Ridge per provinsi (poin pers

### Hasil per kabupaten/kota

`tc_percentage` mempertahankan angka resmi jika tersedia, dan memakai estimasi model terpilih jika kosong. `tc_percentage_prediksi` berisi prediksi model terpilih untuk semua wilayah. Tiga kolom per model memudahkan perbandingan Ridge tabular, Ridge dengan fitur tetangga, dan GCN untuk wilayah yang sama.

`tahun_tc_riwayat` menunjukkan tahun TC resmi terakhir yang tersedia untuk wilayah tersebut. Bila tahun terakhir adalah 2023, model memperkirakan TC 2024 sebagai langkah antara.

In [5]:
results = data25[["tahun", "provinsi", "kabupaten_kota", TARGET]].copy()
results["tc_percentage_prediksi"] = predictions_2025[selected].round(2)
results["tc_percentage"] = results[TARGET].fillna(results["tc_percentage_prediksi"])
results["sumber_tc_percentage"] = np.where(results[TARGET].notna(), "resmi", "estimasi model")
results["tahun_tc_riwayat"] = np.where(official_2024.isna(), 2023, 2024)
results["tc_riwayat_resmi_pct"] = official_2024.fillna(data23[TARGET])
results["model_tc_percentage_prediksi"] = selected
results["tc_percentage_prediksi_ridge"] = predictions_2025["Ridge"].round(2)
results["tc_percentage_prediksi_ridge_tetangga"] = predictions_2025["Ridge + fitur tetangga"].round(2)
results["tc_percentage_prediksi_gcn"] = predictions_2025["GCN"].round(2)

assert results.index.is_unique and len(results) == len(data25)
assert np.isfinite(results.filter(like="tc_percentage_prediksi").drop(
    columns="model_tc_percentage_prediksi"
)).to_numpy().all()
assert results.loc[y25.index, "tc_percentage"].eq(y25).all()
missing_2025 = data25[TARGET].isna()
assert results.loc[missing_2025, "tc_percentage"].eq(
    results.loc[missing_2025, "tc_percentage_prediksi"]
).all()

output = Path("prediksi_tc_2025.csv")
results.to_csv(output, encoding="utf-8-sig")
print(f"{len(results)} wilayah disimpan ke {output}")
print(results.groupby(["sumber_tc_percentage", "tahun_tc_riwayat"]).size().to_string())
print("\nJumlah estimasi per provinsi")
print(results.loc[missing_2025].groupby("provinsi").size().to_string())

154 wilayah disimpan ke prediksi_tc_2025.csv
sumber_tc_percentage  tahun_tc_riwayat
estimasi model        2023                83
                      2024                14
resmi                 2024                57

Jumlah estimasi per provinsi
provinsi
Aceh                         23
Bengkulu                     10
Kepulauan Bangka Belitung     7
Kepulauan Riau                7
Sumatera Selatan             17
Sumatera Utara               33
